<a href="https://colab.research.google.com/github/juanarrow/BigData-notebooks/blob/main/sesion-29.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Sesión 29 - Flume, Sqoop y Oozie

**Módulo:** Big Data Aplicado (BDA - 5075)

---

## Objetivos

- Simular un pipeline de **ingesta de datos** (Flume) con Python
- Simular operaciones **Sqoop** (import/export) entre SQLite y CSV
- Simular un **workflow Oozie** con funciones encadenadas
- Construir un pipeline integrado completo

## 1. Simulación de Sqoop: transferencia RDBMS ↔ HDFS

Sqoop transfiere datos entre bases de datos relacionales y HDFS.
Aquí simulamos ese comportamiento con SQLite (RDBMS) y archivos CSV (HDFS).

In [ ]:
import sqlite3
import pandas as pd
import random
import os
import time
import json
from datetime import datetime, timedelta

random.seed(42)

# === Crear "base de datos MySQL" simulada con SQLite ===
db_tienda = sqlite3.connect(':memory:')
cursor = db_tienda.cursor()

cursor.execute('''
    CREATE TABLE productos (
        id INTEGER PRIMARY KEY,
        nombre TEXT,
        categoria TEXT,
        precio REAL
    )
''')

cursor.execute('''
    CREATE TABLE ventas (
        id INTEGER PRIMARY KEY,
        producto_id INTEGER,
        cantidad INTEGER,
        importe REAL,
        fecha TEXT,
        ciudad TEXT
    )
''')

categorias = ['Electrónica', 'Ropa', 'Hogar', 'Deportes', 'Libros']
prods = [
    ('Portátil', 'Electrónica', 899.99), ('Móvil', 'Electrónica', 599.99),
    ('Camiseta', 'Ropa', 29.99), ('Pantalón', 'Ropa', 49.99),
    ('Lámpara', 'Hogar', 39.99), ('Silla', 'Hogar', 149.99),
    ('Balón', 'Deportes', 24.99), ('Raqueta', 'Deportes', 79.99),
    ('Novela', 'Libros', 15.99), ('Manual', 'Libros', 35.99)
]

for i, (nombre, cat, precio) in enumerate(prods, 1):
    cursor.execute('INSERT INTO productos VALUES (?, ?, ?, ?)', (i, nombre, cat, precio))

ciudades = ['Madrid', 'Barcelona', 'Valencia', 'Sevilla', 'Bilbao']
for i in range(500):
    prod_id = random.randint(1, 10)
    cantidad = random.randint(1, 5)
    precio = prods[prod_id - 1][2]
    fecha = f'2025-{random.randint(1,12):02d}-{random.randint(1,28):02d}'
    cursor.execute('INSERT INTO ventas VALUES (?, ?, ?, ?, ?, ?)',
                   (i+1, prod_id, cantidad, round(precio * cantidad, 2), fecha, random.choice(ciudades)))

db_tienda.commit()

print("=== Base de datos 'tienda' creada ===")
print(f"Productos: {pd.read_sql('SELECT COUNT(*) as n FROM productos', db_tienda).iloc[0,0]}")
print(f"Ventas: {pd.read_sql('SELECT COUNT(*) as n FROM ventas', db_tienda).iloc[0,0]}")
print("\nTabla productos:")
print(pd.read_sql('SELECT * FROM productos', db_tienda).to_string(index=False))

In [ ]:
def sqoop_import(conn, tabla, destino, where=None, num_mappers=4):
    """
    Simula: sqoop import --connect jdbc:... --table <tabla>
            --target-dir <destino> [--where <condicion>]
            --num-mappers <n>
    """
    print(f"sqoop import --table {tabla} --target-dir {destino} ", end='')
    if where:
        print(f"--where \"{where}\" ", end='')
    print(f"--num-mappers {num_mappers}")
    print("-" * 50)

    query = f"SELECT * FROM {tabla}"
    if where:
        query += f" WHERE {where}"

    df = pd.read_sql(query, conn)
    total = len(df)

    os.makedirs(destino, exist_ok=True)

    filas_por_mapper = total // num_mappers
    for i in range(num_mappers):
        inicio = i * filas_por_mapper
        fin = total if i == num_mappers - 1 else (i + 1) * filas_por_mapper
        chunk = df.iloc[inicio:fin]
        archivo = f"{destino}/part-m-{i:05d}.csv"
        chunk.to_csv(archivo, index=False)
        print(f"  Mapper {i}: {len(chunk)} registros → {archivo}")

    print(f"\nImport completado: {total} registros en {num_mappers} partes")
    return total

# === Sqoop Import: tabla completa ===
print("=== SQOOP IMPORT: tabla completa ===\n")
sqoop_import(db_tienda, 'ventas', 'hdfs_simulado/data/ventas')

# === Sqoop Import: con filtro WHERE ===
print(f"\n{'='*50}")
print("=== SQOOP IMPORT: con filtro ===\n")
sqoop_import(db_tienda, 'ventas', 'hdfs_simulado/data/ventas_2025_q1',
             where="fecha >= '2025-01-01' AND fecha < '2025-04-01'",
             num_mappers=2)

In [ ]:
def sqoop_export(conn, tabla_destino, directorio_hdfs, crear_tabla_sql=None):
    """
    Simula: sqoop export --connect jdbc:... --table <tabla>
            --export-dir <directorio>
    """
    print(f"sqoop export --table {tabla_destino} --export-dir {directorio_hdfs}")
    print("-" * 50)

    if crear_tabla_sql:
        conn.execute(crear_tabla_sql)
        conn.commit()

    archivos = [f for f in os.listdir(directorio_hdfs) if f.endswith('.csv')]
    total = 0
    for archivo in sorted(archivos):
        ruta = os.path.join(directorio_hdfs, archivo)
        df = pd.read_csv(ruta)
        df.to_sql(tabla_destino, conn, if_exists='append', index=False)
        total += len(df)
        print(f"  Exportado {archivo}: {len(df)} registros")

    print(f"\nExport completado: {total} registros → tabla '{tabla_destino}'")
    return total

# === Primero, crear resultados procesados en "HDFS" ===
os.makedirs('hdfs_simulado/data/resumen_ventas', exist_ok=True)
resumen = pd.read_sql("""
    SELECT p.categoria, COUNT(*) as num_ventas,
           ROUND(SUM(v.importe), 2) as total,
           ROUND(AVG(v.importe), 2) as media
    FROM ventas v JOIN productos p ON v.producto_id = p.id
    GROUP BY p.categoria
""", db_tienda)
resumen.to_csv('hdfs_simulado/data/resumen_ventas/part-m-00000.csv', index=False)

# === Sqoop Export: HDFS → RDBMS ===
print("=== SQOOP EXPORT: resultados a RDBMS ===\n")
sqoop_export(
    db_tienda,
    'resumen_categorias',
    'hdfs_simulado/data/resumen_ventas',
    crear_tabla_sql="""
        CREATE TABLE IF NOT EXISTS resumen_categorias (
            categoria TEXT, num_ventas INTEGER,
            total REAL, media REAL
        )
    """
)

print("\nVerificación en RDBMS:")
print(pd.read_sql('SELECT * FROM resumen_categorias', db_tienda).to_string(index=False))

## 2. Simulación de Flume: ingesta de datos en streaming

Simulamos un agente Flume con el patrón Source → Channel → Sink usando generadores Python.

In [ ]:
from collections import deque

class FlumeSource:
    """Simula un Flume Source que genera eventos de log"""
    def __init__(self, nombre):
        self.nombre = nombre

    def generar_eventos(self, n=10):
        niveles = ['INFO', 'WARN', 'ERROR', 'DEBUG']
        pesos = [0.6, 0.2, 0.1, 0.1]
        mensajes = {
            'INFO': ['Request processed', 'User logged in', 'Page viewed', 'API call success'],
            'WARN': ['High memory usage', 'Slow query detected', 'Rate limit approaching'],
            'ERROR': ['Connection timeout', 'NullPointerException', 'Disk full', 'Auth failed'],
            'DEBUG': ['Cache hit', 'Query plan', 'Thread pool status']
        }
        eventos = []
        base_time = datetime(2025, 1, 22, 8, 0, 0)
        for i in range(n):
            nivel = random.choices(niveles, weights=pesos, k=1)[0]
            ts = base_time + timedelta(seconds=random.randint(0, 28800))
            evento = {
                'timestamp': ts.strftime('%Y-%m-%d %H:%M:%S'),
                'nivel': nivel,
                'mensaje': random.choice(mensajes[nivel]),
                'servidor': f'web-{random.randint(1,5):02d}',
                'ip': f'192.168.1.{random.randint(1,254)}'
            }
            eventos.append(evento)
        return eventos

class FlumeChannel:
    """Simula un Flume Channel (buffer de eventos)"""
    def __init__(self, nombre, tipo='memory', capacidad=1000):
        self.nombre = nombre
        self.tipo = tipo
        self.capacidad = capacidad
        self.buffer = deque(maxlen=capacidad)
        self.total_recibidos = 0

    def put(self, evento):
        self.buffer.append(evento)
        self.total_recibidos += 1

    def take(self):
        return self.buffer.popleft() if self.buffer else None

    def __len__(self):
        return len(self.buffer)

class FlumeSinkHDFS:
    """Simula un Flume HDFS Sink"""
    def __init__(self, nombre, ruta_base):
        self.nombre = nombre
        self.ruta_base = ruta_base
        self.total_escritos = 0

    def write(self, evento):
        fecha = evento['timestamp'][:10]
        ruta = os.path.join(self.ruta_base, fecha.replace('-', '/'))
        os.makedirs(ruta, exist_ok=True)
        archivo = os.path.join(ruta, 'events.jsonl')
        with open(archivo, 'a') as f:
            f.write(json.dumps(evento) + '\n')
        self.total_escritos += 1

# === Ejecutar pipeline Flume ===
print("=== AGENTE FLUME: Source → Channel → Sink ===")
print("Configuración:")
print("  Source:  Generador de logs de aplicación")
print("  Channel: Memory (capacidad=1000)")
print("  Sink:    HDFS (particionado por fecha)")
print()

source = FlumeSource('log-source')
channel = FlumeChannel('mem-channel', capacidad=1000)
sink = FlumeSinkHDFS('hdfs-sink', 'hdfs_simulado/logs')

eventos = source.generar_eventos(n=100)
print(f"Source generó {len(eventos)} eventos")

for ev in eventos:
    channel.put(ev)
print(f"Channel contiene {len(channel)} eventos (total recibidos: {channel.total_recibidos})")

while len(channel) > 0:
    evento = channel.take()
    sink.write(evento)

print(f"Sink escribió {sink.total_escritos} eventos en HDFS")

print("\nEstructura de archivos generados:")
for root, dirs, files in os.walk('hdfs_simulado/logs'):
    nivel = root.replace('hdfs_simulado/logs', '').count(os.sep)
    indent = '  ' * nivel
    print(f"{indent}{os.path.basename(root)}/")
    for f in files:
        ruta_completa = os.path.join(root, f)
        n_lineas = sum(1 for _ in open(ruta_completa))
        print(f"{indent}  └── {f} ({n_lineas} eventos)")

## 3. Simulación de Oozie: workflow orquestado

Simulamos un workflow Oozie como una secuencia de pasos (funciones) que se ejecutan en orden, con manejo de errores.

In [ ]:
class OozieWorkflow:
    """Simula un workflow de Apache Oozie"""
    def __init__(self, nombre):
        self.nombre = nombre
        self.acciones = []
        self.resultados = {}
        self.estado = 'PREP'

    def add_action(self, nombre, tipo, funcion, on_error=None):
        self.acciones.append({
            'nombre': nombre, 'tipo': tipo,
            'funcion': funcion, 'on_error': on_error
        })

    def run(self):
        print(f"{'='*60}")
        print(f"OOZIE WORKFLOW: {self.nombre}")
        print(f"Estado: RUNNING")
        print(f"Acciones: {len(self.acciones)}")
        print(f"{'='*60}\n")
        self.estado = 'RUNNING'
        inicio_total = time.time()

        for i, accion in enumerate(self.acciones):
            print(f"--- Acción {i+1}/{len(self.acciones)}: [{accion['tipo']}] {accion['nombre']} ---")
            inicio = time.time()
            try:
                resultado = accion['funcion']()
                duracion = time.time() - inicio
                self.resultados[accion['nombre']] = {'estado': 'OK', 'duracion': duracion, 'resultado': resultado}
                print(f"  ✅ OK ({duracion*1000:.0f} ms)\n")
            except Exception as e:
                duracion = time.time() - inicio
                self.resultados[accion['nombre']] = {'estado': 'ERROR', 'duracion': duracion, 'error': str(e)}
                print(f"  ❌ ERROR: {e} ({duracion*1000:.0f} ms)")
                if accion['on_error']:
                    print(f"  Ejecutando handler de error...")
                    accion['on_error'](e)
                self.estado = 'KILLED'
                print(f"\nWorkflow terminado con error.")
                return False

        duracion_total = time.time() - inicio_total
        self.estado = 'SUCCEEDED'
        print(f"{'='*60}")
        print(f"WORKFLOW COMPLETADO: {self.estado}")
        print(f"Duración total: {duracion_total*1000:.0f} ms")
        print(f"{'='*60}")
        return True

# === Definir las acciones del workflow ===
db_workflow = sqlite3.connect(':memory:')

def accion_sqoop_import():
    """Paso 1: Importar datos de RDBMS a HDFS"""
    datos = pd.read_sql('SELECT * FROM ventas', db_tienda)
    os.makedirs('hdfs_simulado/workflow/raw', exist_ok=True)
    datos.to_csv('hdfs_simulado/workflow/raw/ventas.csv', index=False)
    print(f"  Importados {len(datos)} registros de 'ventas'")
    return len(datos)

def accion_pig_etl():
    """Paso 2: Limpiar y transformar con Pig (Pandas)"""
    datos = pd.read_csv('hdfs_simulado/workflow/raw/ventas.csv')
    datos = datos.drop_duplicates()
    datos = datos[datos['importe'] > 0]
    datos['mes'] = datos['fecha'].apply(lambda f: int(f.split('-')[1]))
    datos['trimestre'] = datos['mes'].apply(lambda m: f'Q{(m-1)//3+1}')
    os.makedirs('hdfs_simulado/workflow/clean', exist_ok=True)
    datos.to_csv('hdfs_simulado/workflow/clean/ventas_limpias.csv', index=False)
    print(f"  Registros limpios: {len(datos)}")
    return len(datos)

def accion_hive_load():
    """Paso 3: Cargar en tabla Hive (SQLite)"""
    datos = pd.read_csv('hdfs_simulado/workflow/clean/ventas_limpias.csv')
    datos.to_sql('ventas_limpias', db_workflow, if_exists='replace', index=False)
    count = pd.read_sql('SELECT COUNT(*) as n FROM ventas_limpias', db_workflow).iloc[0,0]
    print(f"  Tabla 'ventas_limpias' creada con {count} registros")
    return count

def accion_hive_query():
    """Paso 4: Ejecutar consultas analíticas"""
    resultado = pd.read_sql("""
        SELECT trimestre, COUNT(*) as ventas,
               ROUND(SUM(importe), 2) as total,
               ROUND(AVG(importe), 2) as media
        FROM ventas_limpias
        GROUP BY trimestre ORDER BY trimestre
    """, db_workflow)
    os.makedirs('hdfs_simulado/workflow/results', exist_ok=True)
    resultado.to_csv('hdfs_simulado/workflow/results/resumen.csv', index=False)
    print(f"  Resultados:")
    print(f"  {resultado.to_string(index=False)}")
    return resultado.to_dict()

def accion_notificacion():
    """Paso 5: Notificar finalización"""
    print(f"  📧 Email enviado a: equipo-datos@empresa.com")
    print(f"  Asunto: ETL Ventas completado exitosamente")
    return True

# === Construir y ejecutar el workflow ===
wf = OozieWorkflow('etl-ventas-diario')
wf.add_action('importar-datos', 'Sqoop', accion_sqoop_import)
wf.add_action('limpiar-datos', 'Pig', accion_pig_etl)
wf.add_action('cargar-hive', 'Hive', accion_hive_load)
wf.add_action('generar-resumen', 'Hive', accion_hive_query)
wf.add_action('notificar', 'Email', accion_notificacion)

wf.run()

## 4. Sqoop incremental: importar solo datos nuevos

En producción, no siempre reimportamos todo. Sqoop permite importar solo los datos nuevos desde la última ejecución.

In [ ]:
def sqoop_import_incremental(conn, tabla, destino, check_column, last_value):
    """
    Simula: sqoop import --incremental append
            --check-column <col> --last-value <val>
    """
    print(f"sqoop import --table {tabla} --incremental append")
    print(f"  --check-column {check_column} --last-value {last_value}")
    print("-" * 50)

    query = f"SELECT * FROM {tabla} WHERE {check_column} > {last_value}"
    df = pd.read_sql(query, conn)

    if len(df) == 0:
        print("  No hay datos nuevos.")
        return 0, last_value

    os.makedirs(destino, exist_ok=True)
    archivo = f"{destino}/part-incr-{last_value}.csv"
    df.to_csv(archivo, index=False)
    nuevo_last = df[check_column].max()
    print(f"  Importados {len(df)} registros nuevos (id > {last_value})")
    print(f"  Nuevo last-value: {nuevo_last}")
    return len(df), nuevo_last

# Simular ejecuciones incrementales
print("=== SQOOP INCREMENTAL: primera ejecución ===\n")
n1, lv1 = sqoop_import_incremental(db_tienda, 'ventas', 'hdfs_simulado/incremental/ventas', 'id', 400)

print(f"\n{'='*50}")
print("=== SQOOP INCREMENTAL: segunda ejecución ===\n")
n2, lv2 = sqoop_import_incremental(db_tienda, 'ventas', 'hdfs_simulado/incremental/ventas', 'id', lv1)

print(f"\nResumen: Primera ejecución importó {n1} registros, segunda {n2} registros")

## 5. Ejercicios propuestos

### Ejercicio 1
Modifica la clase `FlumeSource` para generar eventos de tipo "acceso web" (método HTTP, URL, código de respuesta, tiempo de respuesta). Ejecuta el pipeline completo.

### Ejercicio 2
Añade una nueva tabla `clientes` a la base de datos simulada. Usa `sqoop_import` para importarla y luego haz un JOIN con ventas en el paso Hive del workflow.

### Ejercicio 3
Añade un paso `decision` al workflow que, si el número de errores en los logs supera un umbral, ejecute una acción de alerta en lugar de continuar normalmente.

In [ ]:
# === EJERCICIO 1: Flume con eventos web ===
# Tu código aquí:


# === EJERCICIO 2: Sqoop con tabla clientes ===
# Tu código aquí:


# === EJERCICIO 3: Workflow con decisión ===
# Tu código aquí:


print("Completa los ejercicios arriba.")

## Recursos

- [Apache Flume User Guide](https://flume.apache.org/FlumeUserGuide.html)
- [Apache Sqoop User Guide](https://sqoop.apache.org/docs/1.4.7/SqoopUserGuide.html)
- [Apache Oozie Documentation](https://oozie.apache.org/docs/5.2.1/)

---

[← Sesión 28](./sesion-28.ipynb) | [Índice](../sesiones/index.md) | [Siguiente sesión →](./sesion-30.ipynb)